# Semana 6 · Sesión 2: Visualización y reportes

**Módulo 1**

## Objetivos de la sesión

1. Graficar expresiones simbólicas con `sympy.plotting`, sabiendo qué ofrece y
   dónde se queda corto.
2. Pasar a Matplotlib con `lambdify` cuando hace falta control fino, y
   reconocer el error típico de graficar una expresión con símbolos libres.
3. Generar LaTeX con `sp.latex()` para escribir reportes en los que los
   resultados se calculan en lugar de copiarse a mano.

## Retomamos

En la sesión 1 sacamos, de una matriz de rigidez, las dos frecuencias propias y
los dos modos normales de un par de osciladores acoplados. Todo lo leímos como
fórmulas.

Hoy lo vemos. Y de paso cerramos el Módulo 1: al final de la sesión, un
resultado simbólico puede salir de SymPy y entrar directo a un texto en LaTeX
sin que nadie lo teclee otra vez — que es lo que separa un reporte reproducible
de uno que se desactualiza en cuanto cambias un parámetro.

Este notebook es autocontenido: la celda de abajo vuelve a construir, ya
resuelto, el sistema de la sesión 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

sp.init_printing()

t = sp.Symbol("t", real=True)
x = sp.Symbol("x", real=True)
m, k, A = sp.symbols("m k A", positive=True)

# El sistema de la sesión 1: dos masas iguales, tres resortes iguales.
rigidez = sp.Matrix([[2*k, -k], [-k, 2*k]])
masas = m * sp.eye(2)

# Sus frecuencias propias, con el mismo cálculo de la sesión anterior.
frecuencias = [sp.sqrt(valor) for valor in (masas.inv() * rigidez).eigenvals()]
frecuencias = sorted(frecuencias, key=sp.default_sort_key)

for frecuencia in frecuencias:
    display(frecuencia)

## `sp.plot`: la gráfica en una línea

`sp.plot(expresion, (variable, inicio, fin))` dibuja y ya. Acepta varias
expresiones en la misma llamada, y los adornos usuales como argumentos con
nombre: `title`, `xlabel`, `ylabel`, `legend`.

Es lo más cómodo que hay para mirar rápido una expresión que acabas de
calcular — que es justo lo que uno hace todo el tiempo mientras trabaja.

In [ ]:
# La gráfica se dibuja sola; el resultado se guarda en una variable solo para
# que el notebook no imprima además el objeto que sp.plot devuelve.
grafica_seno = sp.plot(
    sp.sin(x),
    sp.sin(x) - x**3/6 + x,   # el desarrollo de Taylor de orden 3
    (x, -sp.pi, sp.pi),
    title="Seno y su aproximación cúbica",
    xlabel="x",
    ylabel="",
    legend=True,
)

## Otras formas de graficar

La familia completa vive en `sympy.plotting`, y tres de sus funciones se usan
seguido en física:

| Función | Para |
|---|---|
| `sp.plot_parametric(x(t), y(t), (t, a, b))` | Curvas paramétricas: órbitas, retratos de fase |
| `sp.plot_implicit(ecuacion, (x, ...), (y, ...))` | Curvas dadas por una ecuación, sin despejar |
| `sp.plotting.plot3d(f, (x, ...), (y, ...))` | Superficies (`plot3d` no está en el nivel de arriba: hay que pedirlo por `sp.plotting`) |

El **retrato de fase** de un oscilador armónico es el ejemplo canónico de la
primera: con $x = A\cos\omega t$ y $v = -A\omega\sin\omega t$, la curva
$(x(t), v(t))$ tiene que ser una elipse, porque la energía se conserva.

In [ ]:
omega = sp.Symbol("omega", positive=True)

posicion = A * sp.cos(omega * t)
velocidad = sp.diff(posicion, t)

display(sp.Eq(sp.Symbol("v"), velocidad))

# Para graficar hay que fijar números: A y omega son símbolos.
valores = {A: 1, omega: 2}

grafica_fase = sp.plot_parametric(
    posicion.subs(valores),
    velocidad.subs(valores),
    (t, 0, 2*sp.pi),
    title="Retrato de fase del oscilador",
    xlabel="x",
    ylabel="v",
)

## TODO en clase 1

En la semana 5 vimos que el potencial del péndulo, $V(\theta) = mgL(1 -
\cos\theta)$, se parece al de un resorte cuando el ángulo es pequeño. Vamos a
mirar *cuánto* se parece, y hasta dónde.

1. Declara `angulo` como símbolo real y escribe `potencial_exacto`, tomando
   $m = g = L = 1$ para poder graficar (o sustitúyelos después con `subs`).
2. Obtén `potencial_armonico` desarrollando en serie alrededor de cero hasta
   orden 4 y quitando el término $O$ con `.removeO()`.
3. Grafica los dos juntos en una sola llamada a `sp.plot`, de $-\pi$ a $\pi$,
   con `legend=True`.
4. Mirando la gráfica: ¿a partir de qué ángulo, más o menos, la aproximación
   deja de servir? Compáralo con los 5° o 10° que suele citarse en el
   laboratorio.

In [ ]:
# TODO en clase: el potencial del péndulo y su aproximación armónica
angulo = ...

potencial_exacto = ...

potencial_armonico = ...

## Dónde se queda corto `sympy.plotting`

`sp.plot` es cómodo, pero sus opciones se acaban pronto: colores por serie,
rejillas, dos paneles en una figura, anotaciones, guardar en PDF con el tamaño
exacto que pide una revista. Nada de eso está.

Para eso está Matplotlib, y el puente entre los dos mundos ya lo conoces desde
la semana 4: **`lambdify`**, que convierte una expresión simbólica en una
función numérica de verdad. Con `"numpy"` como segundo argumento, la función
resultante acepta arreglos completos y los evalúa de un golpe.

In [ ]:
expresion = sp.exp(-x/4) * sp.sin(2*x)

funcion = sp.lambdify(x, expresion, "numpy")

malla = np.linspace(0, 12, 400)

fig, ax = plt.subplots()
ax.plot(malla, funcion(malla))
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Oscilación amortiguada, graficada con Matplotlib")
ax.grid(True)
plt.show()

## El error clásico: símbolos libres

`lambdify` solo convierte en número lo que sabe evaluar. Si la expresión trae un
símbolo que no está entre los argumentos, la función devuelve un arreglo de
**objetos de SymPy**, no de flotantes — y el error no aparece ahí, sino después,
cuando Matplotlib intenta dibujarlos.

Vale la pena verlo una vez para reconocer el mensaje: `Cannot convert
expression to float`. La receta es siempre la misma: **sustituye todos los
parámetros con `subs` antes de `lambdify`**.

In [ ]:
con_parametro = k * sp.cos(x)          # k quedó suelta

mala = sp.lambdify(x, con_parametro, "numpy")
print("dtype del resultado:", mala(malla).dtype)   # object, no float

try:
    fig, ax = plt.subplots()
    ax.plot(malla, mala(malla))
    fig.canvas.draw()
except TypeError as error:
    print("TypeError:", error)
finally:
    plt.close(fig)

## Física: los dos modos, dibujados

Ahora sí, el sistema de la sesión 1. La solución general es una superposición de
los dos modos:

$$\mathbf{x}(t) = c_1 \begin{pmatrix}1\\1\end{pmatrix}\cos(\omega_1 t)
                + c_2 \begin{pmatrix}-1\\1\end{pmatrix}\cos(\omega_2 t)$$

Elegimos una condición inicial concreta: **jalamos solo la primera masa** hasta
$A$ y soltamos las dos desde el reposo, o sea $\mathbf{x}(0) = (A, 0)$. Eso fija
$c_1$ y $c_2$ con un sistema lineal de 2×2 — de los de la sesión pasada.

In [ ]:
c1, c2 = sp.symbols("c1 c2", real=True)

modo_simetrico = sp.Matrix([1, 1])
modo_antisimetrico = sp.Matrix([-1, 1])

omega1, omega2 = frecuencias

movimiento = (
    c1 * modo_simetrico * sp.cos(omega1*t)
    + c2 * modo_antisimetrico * sp.cos(omega2*t)
)

coeficientes = sp.solve(
    sp.Eq(movimiento.subs(t, 0), sp.Matrix([A, 0])),
    [c1, c2],
    dict=True,
)[0]

movimiento = sp.simplify(movimiento.subs(coeficientes))
display(sp.Eq(sp.Symbol("x(t)"), movimiento, evaluate=False))

Antes de graficar, la costumbre de la semana 5: verificar. La solución debe
cumplir $M\ddot{\mathbf{x}} = -K\mathbf{x}$, así que el residuo tiene que ser
la matriz cero.

In [ ]:
residuo = sp.simplify(masas * sp.diff(movimiento, t, 2) + rigidez * movimiento)

display(sp.Eq(sp.Symbol("residuo"), residuo, evaluate=False))

In [ ]:
numeros = {k: 1, m: 1, A: 1}

x1 = sp.lambdify(t, movimiento[0].subs(numeros), "numpy")
x2 = sp.lambdify(t, movimiento[1].subs(numeros), "numpy")

tiempos = np.linspace(0, 20, 800)

fig, ax = plt.subplots()
ax.plot(tiempos, x1(tiempos), label="masa 1")
ax.plot(tiempos, x2(tiempos), label="masa 2")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("t")
ax.set_ylabel("desplazamiento")
ax.set_title("Superposición de los dos modos normales")
ax.legend()
plt.show()

Ninguna de las dos curvas es un coseno: son la suma de dos, con frecuencias
$\sqrt{k/m}$ y $\sqrt{3k/m}$. La energía va y viene entre las masas, pero de
forma complicada, porque las dos frecuencias no se parecen.

## TODO en clase 2

Cuando las dos frecuencias sí se parecen aparece un fenómeno con nombre propio:
los **batidos**. Para conseguirlo basta debilitar el resorte de en medio.

Con constantes $k$ en los resortes de las paredes y $k_c \ll k$ en el central,
la matriz de rigidez es

$$K_c = \begin{pmatrix} k + k_c & -k_c \\ -k_c & k + k_c \end{pmatrix}$$

1. Declara `k_acoplamiento` como símbolo positivo y arma `rigidez_debil`.
2. Saca sus dos frecuencias como en la sesión 1. Deben salir $\sqrt{k/m}$ y
   $\sqrt{(k + 2k_c)/m}$: la primera no cambia, porque en el modo simétrico el
   resorte central no trabaja.
3. Repite la construcción de `movimiento` con la misma condición inicial
   $(A, 0)$, sustituye `{k: 1, m: 1, A: 1, k_acoplamiento: sp.Rational(1, 20)}`
   y grafica las dos masas con `lambdify` y Matplotlib de $t = 0$ a $t = 200$.
4. Mide en la gráfica el periodo con el que la energía pasa de una masa a la
   otra y compáralo con $2\pi/(\omega_2 - \omega_1)$.

In [ ]:
# TODO en clase: acoplamiento débil y batidos
k_acoplamiento = ...

rigidez_debil = ...

frecuencias_debiles = ...

## `sp.latex`: del resultado al reporte

`sp.latex(expresion)` devuelve una **cadena** con el código LaTeX de la
expresión. Es una cadena de Python y no una expresión de SymPy, así que esta sí
va con `print`: lo interesante es el texto literal.

Es exactamente lo que `sp.init_printing()` viene usando por debajo desde la
semana 4 para que las expresiones se vean bien en el notebook. La diferencia es
que ahora nos quedamos con la cadena para ponerla donde queramos.

In [ ]:
print(sp.latex(omega2))
print(sp.latex(movimiento[0]))

# Y para pegar en un documento, con el entorno de ecuación ya puesto:
print(sp.latex(sp.Eq(sp.Symbol("omega_2"), omega2), mode="equation"))

## Un reporte que se recalcula solo

Con esa cadena se puede escribir texto que **contiene** resultados calculados.
`Markdown` de `IPython.display` interpreta el markdown y su matemática, así que
una celda de código puede producir un párrafo con las fórmulas ya dentro.

La ventaja no es estética: si cambias el sistema y vuelves a ejecutar, el
párrafo cambia con él. Nada que copiar a mano, nada que se quede desactualizado.

In [ ]:
from IPython.display import Markdown

reporte = f"""
El sistema de dos masas acopladas tiene dos frecuencias propias,

$$\\omega_1 = {sp.latex(omega1)}, \\qquad \\omega_2 = {sp.latex(omega2)}$$

y con la condición inicial $(A, 0)$ la primera masa se mueve como

$$x_1(t) = {sp.latex(movimiento[0])}$$
"""

display(Markdown(reporte))

## TODO en clase 3

Cierra el reporte con lo que calculaste en el TODO 2.

1. Escribe una cadena de f-string que enuncie las dos frecuencias del sistema
   con acoplamiento débil, insertándolas con `sp.latex`.
2. Agrega una línea con el **cociente** $\omega_2/\omega_1$ simplificado, que es
   el número que decide si hay batidos visibles o no.
3. Muéstralo con `display(Markdown(...))`.

Un detalle de sintaxis: dentro de una f-string, las llaves de LaTeX hay que
duplicarlas (`{{` y `}}`), porque las sencillas las usa Python para interpolar.
Y las diagonales invertidas van dobles (`\\omega`), como en la celda de arriba.

In [ ]:
# TODO en clase: el reporte del sistema con acoplamiento débil
reporte_debil = ...

## Resumen

Hoy le pusimos imagen y texto a lo simbólico. `sp.plot` grafica en una línea, y
`plot_parametric`, `plot_implicit` y `sp.plotting.plot3d` cubren curvas
paramétricas, curvas implícitas y superficies. Cuando eso se queda corto,
`lambdify` con `"numpy"` entrega una función numérica y Matplotlib hace el
resto — recordando siempre sustituir los parámetros **antes**, porque una
expresión con símbolos libres se rompe al dibujarla.

`sp.latex()` cierra el círculo: el resultado que SymPy calculó entra a un
párrafo sin que nadie lo teclee, y el reporte se recalcula solo al cambiar los
datos.

Con esto termina el **Módulo 1**. En cuatro semanas pasamos de declarar un
símbolo a resolver, diagonalizar, graficar y reportar un sistema físico
completo.

**Tarea de la semana:** [`tarea/tarea-06.ipynb`](../tarea/tarea-06.ipynb) — es
además el **entregable del Módulo 1**, así que pesa más y se evalúa también la
documentación y el historial de commits. Léela pronto y trabájala en varios
commits, no en uno solo la noche anterior.

**Próxima semana — Semana 7:** empieza el Módulo 2, SymPy aplicado a la física.
Entramos a `sympy.physics.mechanics`: marcos de referencia, vectores y
cinemática, donde las matrices de hoy reaparecen como rotaciones.